# Amanda Luijendijk — Exploratory K means clustering
This extra analysis follows the course clustering notebook. It looks for response profiles in awareness of feed personalisation and perceived control. It does not replace the SQ2 hypothesis tests or SEM.

We use four observed survey items: awareness, control, influence over the feed, and the reverse-coded algorithm-decides item. These are separate features, not a validated control scale. We exclude age from clustering so that it does not define the clusters. Age is used only to describe them afterwards. No names, emails or IDs enter the model.

We compare k = 2 to 10 without fixing the cluster count in advance. We inspect the elbow plot, silhouette score (higher is better), Davies–Bouldin index (lower is better), cluster sizes and profiles. We report the highest-silhouette solution as a provisional descriptive choice among these candidates, not a uniquely optimal number of real groups. The k = 7 value in the course marketing example does not carry over to our survey.

K means uses Euclidean distances and treats answer steps as equally spaced. That is an approximation for ordinal survey answers. Z-scaling gives each item equal variance weight; it does not make responses normal. Clusters can depend on the items, coding, scaling and random start. Prepared with Codex assistance.


In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score

cluster_paths = [Path(os.environ.get('BS_DATA_PATH', 'Responses.xlsx')), Path('/content/drive/MyDrive/Colab Notebooks/Responses.xlsx'), Path.home() / 'Downloads' / 'Responses (3).xlsx']
cluster_path = next((p for p in cluster_paths if p.is_file()), None)
if cluster_path is None:
    raise FileNotFoundError('Place Responses.xlsx beside this notebook, mount Google Drive, or set BS_DATA_PATH.')
cluster_raw = pd.read_excel(cluster_path)
cluster_raw.columns = cluster_raw.columns.str.replace('\u00a0', ' ', regex=False).str.strip()
cluster_questions = {
    'awareness': 'I am informed that my feed is shaped by personal information like my age, gender or location - transparency',
    'control': 'I feel I am in control of my feed',
    'influence': 'I can influence the type of content I get on my feed',
    'less_algorithm_control': 'The algorithm decides what I watch',
}
cluster_scale = {'Strongly disagree': 1, 'Disagree': 2, 'Neutral': 3, 'Agree': 4, 'Strongly agree': 5}
cluster_data = pd.DataFrame(index=cluster_raw.index)
for short, header in cluster_questions.items():
    answers = cluster_raw[header].astype('string').str.strip().replace('', pd.NA)
    assert answers.dropna().isin(cluster_scale).all(), f'Unknown labels in {short}'
    cluster_data[short] = answers.map(cluster_scale).astype(float)
cluster_data['less_algorithm_control'] = 6 - cluster_data['less_algorithm_control']
cluster_data = cluster_data.dropna()
cluster_X = StandardScaler().fit_transform(cluster_data)
print('Included:', len(cluster_data), '| Excluded:', len(cluster_raw) - len(cluster_data))
print('Distinct answer profiles:', len(cluster_data.drop_duplicates()))


## Amanda Luijendijk — Compare possible cluster counts
Inertia always tends to fall when k increases, so its smallest value alone is not a reason to choose the largest k. Look for a bend in the elbow curve, and compare it with separation and cluster sizes. Fixed seeds and n_init = 20 make the runs reproducible and reduce sensitivity to a single start.


In [ ]:
cluster_models = {}
cluster_rows = []
for k in range(2, 11):
    candidate = KMeans(n_clusters=k, random_state=20261102, n_init=20).fit(cluster_X)
    cluster_models[k] = candidate
    sizes = np.bincount(candidate.labels_, minlength=k)
    cluster_rows.append({'k': k, 'inertia': candidate.inertia_, 'silhouette': silhouette_score(cluster_X, candidate.labels_), 'DBI': davies_bouldin_score(cluster_X, candidate.labels_), 'smallest_cluster': sizes.min()})
cluster_metrics = pd.DataFrame(cluster_rows).set_index('k')
display(cluster_metrics.round(4))
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, metric in zip(axes, ['inertia', 'silhouette', 'DBI']):
    ax.plot(cluster_metrics.index, cluster_metrics[metric], marker='o')
    ax.set(xlabel='Number of clusters k', ylabel=metric, xticks=cluster_metrics.index)
plt.tight_layout()
plt.show()
cluster_best_k = int(cluster_metrics['silhouette'].idxmax())
print('Highest silhouette among tested candidates:', cluster_best_k)


## Amanda Luijendijk — Profiles and age descriptions
The table shows mean original item scores (1–5) and cluster sizes. Higher less_algorithm_control means less agreement that the algorithm decides what the respondent watches. Cluster numbers are arbitrary labels, not an ordered ranking. Differences in these features are built into clustering and must not be presented as independent hypothesis-test discoveries.

We show the provisional highest-silhouette solution. Percentages by age describe the available sample; they are not a test that age causes cluster membership. Missing ages are reported separately.

**Simple charts:** The first chart shows how many people are in each cluster. The second compares mean awareness and control scores. These are descriptions of the clusters, not new hypothesis tests. The other two features also helped form the clusters. Taking means treats answer steps as equally spaced. Cluster numbers are labels, not a ranking.


In [ ]:
cluster_age_order = ['Under 18', '18-24', '25-34', '35-44', '45-54', '55-64', '65 and older']
for k in [cluster_best_k]:
    labelled = cluster_data.copy()
    labelled['cluster'] = cluster_models[k].labels_ + 1
    profile = labelled.groupby('cluster')[list(cluster_questions)].mean()
    profile.insert(0, 'n', labelled.groupby('cluster').size())
    print(f'k = {k}: mean item profiles')
    display(profile.round(3))
    ages = cluster_raw.loc[labelled.index, 'What age group are you in?'].astype('string').str.strip().fillna('Missing age')
    age_counts = pd.crosstab(labelled['cluster'], ages).reindex(columns=cluster_age_order + ['Missing age'], fill_value=0)
    print('Age counts within each cluster')
    display(age_counts)
    print('Age percentages within each cluster, including missing age')
    display(age_counts.div(age_counts.sum(axis=1), axis=0).mul(100).round(1))
    # Simple chart 1: the number of people in each cluster.
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(profile.index.astype(str), profile['n'], color='steelblue')
    ax.bar_label(bars, labels=[str(int(v)) for v in profile['n']], padding=3)
    ax.set(xlabel='Cluster', ylabel='Number of people', title=f'Amanda Luijendijk: cluster sizes (k = {k})')
    ax.set_ylim(0, profile['n'].max() * 1.18)
    plt.tight_layout()
    plt.show()

    # Simple chart 2: the two main SQ2 scores in each cluster.
    ax = profile[['awareness', 'control']].plot.bar(figsize=(9, 4), color=['steelblue', 'darkorange'], rot=0)
    ax.set(xlabel='Cluster', ylabel='Mean score (1–5)', title='Amanda Luijendijk: awareness and control by cluster', ylim=(0, 5.6))
    ax.set_yticks([0, 1, 2, 3, 4, 5])
    ax.legend(['Awareness', 'Control'], loc='upper left', bbox_to_anchor=(0, 1.02), ncol=2)
    plt.tight_layout()
    plt.show()


## Amanda Luijendijk — Random start check and conclusion
We refit the provisional selected k with five other seeds. Adjusted Rand Index compares cluster membership without relying on label numbers. A value of 1 means identical assignments. This checks random starts on the same sample; it is not evidence of stability in new respondents or with different items.


In [ ]:
cluster_aris = []
for seed in [11, 22, 33, 44, 55]:
    repeat = KMeans(n_clusters=cluster_best_k, random_state=seed, n_init=20).fit(cluster_X)
    cluster_aris.append(adjusted_rand_score(cluster_models[cluster_best_k].labels_, repeat.labels_))
display(pd.DataFrame({'seed': [11, 22, 33, 44, 55], 'adjusted_rand_index': cluster_aris}).round(3))
best_row = cluster_metrics.loc[cluster_best_k]
display(Markdown(f"""### Amanda Luijendijk — Clustering result in simple English
We clustered **{len(cluster_data)} respondents** using four observed answer scores. Among k = 2 to 10, **k = {cluster_best_k}** had the highest silhouette score (**{best_row.silhouette:.3f}**; DBI **{best_row.DBI:.3f}**; smallest cluster **{int(best_row.smallest_cluster)}**).

Across the five extra random seeds, the selected solution's Adjusted Rand Index ranged from **{min(cluster_aris):.3f} to {max(cluster_aris):.3f}**. This is a same-sample random-start check only.

These clusters are exploratory response profiles. They depend on our features and equal-spacing assumption. They do not prove that natural population groups exist, validate the weak control scale, or answer causal questions. Age is described afterwards and did not determine the clusters. No new confirmatory p-values are produced by this clustering step.

**Source:** course notebook Clustering-with-Kmeans-DBSCAN-HDBSCAN, K-means section. The feature selection and explicit n_init, seed checks and k range are adapted for our survey. DBSCAN and HDBSCAN have not been fitted in this section.
"""))
